# 06 — Wider graph: higher cointegration rank and deterministic terms

The first four notebooks built up to a fully working `BayesianVECM` — but only for the narrowest possible v0 envelope: `coint_rank=1` and `deterministic="n"`. This notebook shows what changes now that the graph supports the full v0 scope.

Two extensions land in this slice:

1. **Higher cointegration rank ($r > 1$).** When $K$ variables share more than one long-run relationship the model needs $r$ cointegrating vectors — a $(K, r)$ matrix $\beta$ instead of a $(K, 1)$ column. The Johansen identification generalises: the top $r \times r$ block of $\beta$ is pinned to $I_r$, and the remaining $(K - r) \times r$ entries are free.

2. **Deterministic terms.** Real data usually has a non-zero mean or a trend; ignoring that pushes everything into the residuals and biases the cointegration estimates. The design layer already supported five codes (`"n"`, `"co"`, `"ci"`, `"lo"`, `"li"`); now the PyMC graph does too.

The implementation insight is simple: the graph reads shapes *off the design matrices* rather than reconstructing them from `k_ar_diff` and `deterministic`. Because `cointegration_design` already appended the right deterministic columns, the graph gets the right shapes for free — no separate code path per code.

**What changes in the model:**

| Code | `y_lag1` cols | `delta_x` cols | Effect on graph |
|------|--------------|----------------|----------------|
| `"n"` | $K$ | $Kk$ | baseline |
| `"co"` | $K$ | $Kk + 1$ | $\Gamma$ widens by one column |
| `"lo"` | $K$ | $Kk + 1$ | $\Gamma$ widens by one column |
| `"ci"` | $K + 1$ | $Kk$ | $\beta$ gains one free row |
| `"li"` | $K + 1$ | $Kk$ | $\beta$ gains one free row |

In [ ]:
from __future__ import annotations

import numpy as np

from bayesian_vecm import BayesianVECM
from bayesian_vecm._design import cointegration_design

np.set_printoptions(precision=3, suppress=True)

In [ ]:
# ---------------------------------------------------------------------------
# Sampling configuration
# ---------------------------------------------------------------------------
# Set FAST_SAMPLING = True for quick execution (CI / first read-through).
# Set FAST_SAMPLING = False for publication-quality posteriors.
# ---------------------------------------------------------------------------
FAST_SAMPLING = True

if FAST_SAMPLING:
    DRAWS, TUNE, CHAINS = 200, 200, 2
else:
    DRAWS, TUNE, CHAINS = 1000, 1000, 4

print(f"Sampling config: draws={DRAWS}, tune={TUNE}, chains={CHAINS}")
if FAST_SAMPLING:
    print("(FAST_SAMPLING=True — posteriors are coarser; set to False for full quality)")

## 1. Higher cointegration rank: a trivariate $r = 2$ model

### The econometrics

With $K$ variables the cointegration rank $r$ tells us how many independent long-run equilibria exist among them. $r = 1$ (notebooks 03–05) is the simplest case: one cointegrating vector, one error-correction term. $r = 2$ means *two* linearly independent long-run relationships.

Everything in the VECM equation generalises:

$$
\Delta y_t = \underbrace{\alpha}_{K \times r}\,\underbrace{\beta^{\top}}_{r \times K}\,y_{t-1}
           + \Gamma\,\Delta x_t + \varepsilon_t.
$$

- $\beta$ is $(K, r)$ — each column is one cointegrating vector.
- $\alpha$ is $(K, r)$ — each column is the loadings on the corresponding relation.
- Identification: pin $\beta[:r, :] = I_r$. For $r = 2$ the top $2 \times 2$ block is fixed at $I_2$.

### The DGP

We generate a trivariate series ($K = 3$) with two cointegrating relations:

$$
\beta_1 = (1, 0, -0.5)^{\top}, \qquad \beta_2 = (0, 1, -0.3)^{\top}.
$$

In the Johansen normalisation these are stacked as columns of $\beta$:

$$
\beta = \begin{pmatrix} 1 & 0 \\ 0 & 1 \\ -0.5 & -0.3 \end{pmatrix}.
$$

The top $2 \times 2$ block is $I_2$ — exactly the pin the graph enforces. The free row is $(-0.5, -0.3)$ and is what the sampler estimates.

In [ ]:
def make_trivariate_r2(
    n_obs: int = 150,
    seed: int = 7,
) -> np.ndarray:
    """Trivariate VECM DGP with cointegration rank 2.

    Two cointegrating relations:
      ec1 = y0 - 0.5 * y2
      ec2 = y1 - 0.3 * y2

    True beta = [[1, 0], [0, 1], [-0.5, -0.3]]  (Johansen-normalised).
    True alpha = [[-0.3, 0], [0, -0.3], [0.1, 0.1]].
    """
    rng = np.random.default_rng(seed=seed)
    y = np.zeros((n_obs, 3))
    y[0] = rng.normal(size=3)
    for t in range(1, n_obs):
        ec1 = y[t - 1, 0] - 0.5 * y[t - 1, 2]
        ec2 = y[t - 1, 1] - 0.3 * y[t - 1, 2]
        y[t, 0] = y[t - 1, 0] - 0.3 * ec1 + rng.normal(scale=0.5)
        y[t, 1] = y[t - 1, 1] - 0.3 * ec2 + rng.normal(scale=0.5)
        y[t, 2] = y[t - 1, 2] + 0.1 * ec1 + 0.1 * ec2 + rng.normal(scale=0.5)
    return y


endog_r2 = make_trivariate_r2(n_obs=150, seed=7)
print(f"Shape: {endog_r2.shape}  (T=150, K=3)")
print("\nFirst 5 rows:")
print(endog_r2[:5])

## 2. Fit the $r = 2$ model

The public API is identical to the $r = 1$ case — just pass `coint_rank=2`. The graph builds a $(3, 2)$ $\beta$ with the top $2 \times 2$ block pinned to $I_2$ and the bottom row free.

In [ ]:
model_r2 = BayesianVECM(k_ar_diff=1, coint_rank=2, deterministic="n")
model_r2.fit(
    endog_r2,
    draws=DRAWS,
    tune=TUNE,
    chains=CHAINS,
    random_seed=42,
    progressbar=False,
    cores=1,
)

print("Fit complete.")
print(model_r2.summary())

## 3. Inspecting $\beta$: shape and identification

Three things to verify:

1. **Shape.** `beta` should be $(3, 2)$ — three variables, two cointegrating vectors.
2. **Pin.** The top $2 \times 2$ block should be exactly $I_2$ in every posterior draw.
3. **Free row.** The third row should recover something close to $(-0.5, -0.3)$ — the true values we put in the DGP.

In [ ]:
beta_draws = model_r2.idata_.posterior["beta"].values  # (chains, draws, K, r)
n_chains, n_draws, K, r = beta_draws.shape
print(f"beta shape: ({K}, {r})  — (K, r) as expected")
print()

# 1. Pin: take one draw and check the top 2x2 block
one_draw = beta_draws[0, 0]
print("One posterior draw of beta:")
print(one_draw)
print()
print("Top 2x2 block (should be I_2 exactly):")
print(one_draw[:r, :])
print()

# 2. Free row: posterior mean
free_row_mean = beta_draws[:, :, 2, :].mean(axis=(0, 1))
print(f"Free row (beta[2, :]) — posterior mean : {free_row_mean}")
print("Free row (beta[2, :]) — true values    : [-0.5, -0.3]")

## 4. Inspecting $\alpha$: loading matrix

$\alpha$ is $(K, r) = (3, 2)$. Column $j$ of $\alpha$ gives the speed-of-adjustment coefficients for cointegrating relation $j$. The true values are:

$$
\alpha = \begin{pmatrix} -0.3 & 0 \\ 0 & -0.3 \\ 0.1 & 0.1 \end{pmatrix}.
$$

With 150 observations and fast sampling the posteriors won't be tight, but the signs should be right and the magnitudes in the right ballpark.

In [ ]:
alpha_draws = model_r2.idata_.posterior["alpha"].values  # (chains, draws, K, r)
alpha_mean = alpha_draws.mean(axis=(0, 1))
alpha_sd = alpha_draws.std(axis=(0, 1))

print(f"alpha shape: {alpha_mean.shape}  — (K, r) = (3, 2)")
print()
print("Posterior mean of alpha:")
print(alpha_mean)
print()
print("Posterior std of alpha:")
print(alpha_sd)
print()
print("True alpha:")
print(np.array([[-0.3, 0.0], [0.0, -0.3], [0.1, 0.1]]))

## 5. Deterministic terms: inside vs outside the cointegrating relation

Most real-world time series have a non-zero unconditional mean, and many have a drift. A VECM without deterministic terms forces the cointegrating relation to have mean zero and the levels to have no drift — usually wrong.

The five codes in v0 correspond to Johansen's (1995) five cases:

| Code | Term | Position | Johansen case |
|------|------|----------|---------------|
| `"n"` | none | — | 1 |
| `"co"` | constant | outside — in $\Gamma\,\Delta x_t$ | 2 |
| `"ci"` | constant | inside — in $\beta^{\top} y_{t-1}$ | 3 |
| `"lo"` | linear trend | outside — in $\Gamma\,\Delta x_t$ | 4 (partial) |
| `"li"` | linear trend | inside — in $\beta^{\top} y_{t-1}$ | 5 (partial) |

**Inside vs outside — the intuition:**

- **Outside (`"co"`, `"lo"`):** the constant or trend enters the short-run equation directly, like adding an intercept to an OLS regression. The cointegrating relation itself is still assumed to have mean zero. $\Delta x$ gains one column, so $\Gamma$ widens from $(K, Kk)$ to $(K, Kk+1)$.

- **Inside (`"ci"`, `"li"`):** the constant or trend is *restricted to the cointegrating relation* — it shifts the long-run equilibrium rather than just the short-run dynamics. This is usually the economically appropriate choice when you believe the series are cointegrated around a non-zero mean. $y_{t-1}$ gains one column, so $\beta$ widens from $(K, r)$ to $(K+1, r)$ and the extra row is free (sampled).

**How to choose:** if the cointegrating relation itself has a non-zero mean (e.g. a spread that trades around 50 basis points, not zero), use `"ci"`. If the levels drift but the cointegrating relation is mean-zero, use `"co"`. In practice `"ci"` (Johansen case 3) is the most common default.

## 6. Inside constant (`"ci"`): $\beta$ gains a free row

We use the bivariate DGP from notebook 05 — $\beta = (1, -0.5)^{\top}$, $\alpha = (-0.4, 0.2)^{\top}$ — but add a non-zero mean to the cointegrating relation. That is, the true long-run equilibrium is $y_0 - 0.5 y_1 + \mu = 0$ for some intercept $\mu$, which `"ci"` can estimate.

With `deterministic="ci"`, `cointegration_design` appends a column of ones to `y_lag1`, making it $(T_{\text{eff}}, K+1)$. The graph reads that shape and builds $\beta$ as $(K+1, r) = (3, 1)$. The first entry is still pinned to 1; the second (the $y_1$ coefficient) and the third (the intercept) are both free.

In [ ]:
def make_bivariate_ci(
    n_obs: int = 120,
    mu_ci: float = 2.0,  # intercept inside the cointegrating relation
    seed: int = 1,
) -> np.ndarray:
    """Bivariate VECM DGP with a constant inside the cointegrating relation.

    Long-run equilibrium: y0 - 0.5 * y1 + 2.0 = 0, i.e. y0 ≈ 0.5 * y1 - 2.
    True beta (Johansen) = [1, -0.5, 2.0]^T  (length-3 vector for K+1=3).
    alpha = [-0.4, 0.2].
    """
    rng = np.random.default_rng(seed=seed)
    y = np.zeros((n_obs, 2))
    y[0] = rng.normal(size=2)
    for t in range(1, n_obs):
        # ec includes the intercept inside the cointegrating relation
        ec = y[t - 1, 0] - 0.5 * y[t - 1, 1] + mu_ci
        y[t, 0] = y[t - 1, 0] - 0.4 * ec + rng.normal(scale=0.5)
        y[t, 1] = y[t - 1, 1] + 0.2 * ec + rng.normal(scale=0.5)
    return y


endog_ci = make_bivariate_ci(n_obs=120, mu_ci=2.0, seed=1)
print(f"Shape: {endog_ci.shape}")
print("\nSample mean (both series should have non-zero mean):")
print(endog_ci.mean(axis=0))

In [ ]:
# Inspect the design to confirm y_lag1 has K+1 = 3 columns.
design_ci = cointegration_design(endog_ci, k_ar_diff=1, deterministic="ci")
print(f"delta_y shape : {design_ci.delta_y.shape}   (T_eff, K)")
print(f"delta_x shape : {design_ci.delta_x.shape}   (T_eff, K*k)  — unchanged")
print(f"y_lag1 shape  : {design_ci.y_lag1.shape}  (T_eff, K+1) — extra column is the ones column")
print()
print("Last column of y_lag1 (should be all ones):")
print(design_ci.y_lag1[:5, -1])

In [ ]:
model_ci = BayesianVECM(k_ar_diff=1, coint_rank=1, deterministic="ci")
model_ci.fit(
    endog_ci,
    draws=DRAWS,
    tune=TUNE,
    chains=CHAINS,
    random_seed=42,
    progressbar=False,
    cores=1,
)

print("Fit complete.")
print(model_ci.summary())

In [ ]:
beta_ci = model_ci.idata_.posterior["beta"].values  # (chains, draws, K+1, r)
print(f"beta shape: {beta_ci.shape[2:]}  — (K+1, r) = (3, 1) as expected")
print()

beta_ci_mean = beta_ci.mean(axis=(0, 1))  # (3, 1)
print("Posterior mean of beta (K+1, r):")
print(beta_ci_mean)
print()
print("Interpretation:")
print(f"  beta[0, 0] = {beta_ci_mean[0, 0]:.4f}  (pinned to 1.0 — Johansen normalisation)")
print(f"  beta[1, 0] = {beta_ci_mean[1, 0]:.4f}  (y1 coefficient; true = -0.5)")
print(f"  beta[2, 0] = {beta_ci_mean[2, 0]:.4f}  (intercept inside EC; true = 2.0)")

## 7. Outside constant (`"co"`): $\Gamma$ gains a column

With `deterministic="co"` the constant enters the short-run equation but *not* the cointegrating relation — the long-run equilibrium is still assumed mean-zero. `cointegration_design` appends a column of ones to `delta_x`, making it $(T_{\text{eff}}, Kk+1)$. The graph reads that shape and builds $\Gamma$ as $(K, Kk+1)$ — one extra column for the intercept of the short-run dynamics.

$\beta$ stays $(K, r) = (2, 1)$ because `y_lag1` is unchanged.

We reuse the same DGP as notebooks 04–05 (mean-zero cointegrating relation), which is the right setting for `"co"`.

In [ ]:
def make_bivariate_baseline(
    n_obs: int = 120,
    seed: int = 2,
) -> np.ndarray:
    """Bivariate VECM DGP with mean-zero cointegrating relation.

    Same as notebooks 04-05: beta=(1, -0.5), alpha=(-0.4, 0.2).
    """
    rng = np.random.default_rng(seed=seed)
    y = np.zeros((n_obs, 2))
    y[0] = rng.normal(size=2)
    for t in range(1, n_obs):
        ec = y[t - 1, 0] - 0.5 * y[t - 1, 1]
        y[t, 0] = y[t - 1, 0] - 0.4 * ec + rng.normal(scale=0.5)
        y[t, 1] = y[t - 1, 1] + 0.2 * ec + rng.normal(scale=0.5)
    return y


endog_co = make_bivariate_baseline(n_obs=120, seed=2)

# Inspect design
design_co = cointegration_design(endog_co, k_ar_diff=1, deterministic="co")
print(f"delta_y shape : {design_co.delta_y.shape}   (T_eff, K)")
extra = "(extra column is the outside-term ones column)"
print(f"delta_x shape : {design_co.delta_x.shape}  (T_eff, K*k+1) {extra}")
print(f"y_lag1 shape  : {design_co.y_lag1.shape}   (T_eff, K)  — unchanged")
print()
print("Last column of delta_x (should be all ones):")
print(design_co.delta_x[:5, -1])

In [ ]:
model_co = BayesianVECM(k_ar_diff=1, coint_rank=1, deterministic="co")
model_co.fit(
    endog_co,
    draws=DRAWS,
    tune=TUNE,
    chains=CHAINS,
    random_seed=42,
    progressbar=False,
    cores=1,
)

print("Fit complete.")
print(model_co.summary())

In [ ]:
gamma_co = model_co.idata_.posterior["Gamma"].values  # (chains, draws, K, Kk+1)
beta_co = model_co.idata_.posterior["beta"].values  # (chains, draws, K, r)

print(f"Gamma shape: {gamma_co.shape[2:]}  — (K, Kk+1) = (2, 3) as expected")
print(f"beta shape : {beta_co.shape[2:]}   — (K, r) = (2, 1) — unchanged by outside term")
print()

gamma_mean = gamma_co.mean(axis=(0, 1))
print("Posterior mean of Gamma (K, Kk+1):")
print(gamma_mean)
print()
print("Interpretation:")
print("  Columns 0-1: short-run lag coefficients (Gamma_1)")
print(f"  Column 2   : intercept of the short-run equation (posterior mean = {gamma_mean[:, 2]})")
print()
beta_co_mean = beta_co.mean(axis=(0, 1))
print(f"beta[1, 0] posterior mean: {beta_co_mean[1, 0]:.4f}  (true = -0.5)")

## 8. What this unlocks — and what's still to come

**Shipped this slice (`feat/wider-graph`):**

The PyMC graph now covers the full v0 envelope:

```python
# Higher rank:
model = BayesianVECM(k_ar_diff=1, coint_rank=2, deterministic="n")

# Constant inside the cointegrating relation:
model = BayesianVECM(k_ar_diff=1, coint_rank=1, deterministic="ci")

# Constant outside (in short-run dynamics):
model = BayesianVECM(k_ar_diff=1, coint_rank=1, deterministic="co")

# Trend codes work the same way:
model = BayesianVECM(k_ar_diff=1, coint_rank=1, deterministic="li")
model = BayesianVECM(k_ar_diff=1, coint_rank=1, deterministic="lo")
```

All five codes work for any valid `coint_rank`. The key insight — reading shapes off the design matrices rather than branching on `deterministic` — kept the implementation to a handful of lines.

**Still to come (modelling extensions, in rough order):**

- **Sparse priors (horseshoe).** The $\Gamma$ block has $K^2 k$ parameters; most are near zero in practice. A regularised horseshoe would shrink irrelevant entries automatically.
- **Stochastic volatility.** Replace constant $\Sigma$ with a time-varying Cholesky structure — important for financial data.
- **Uncertain cointegration rank.** Fix $r$ is a deliberate simplification; inferring it jointly is harder and deferred until the fixed-$r$ estimator is solid.